
# Face Preprocessing

## Objective
The objective of this notebook is to remove distracting backgrounds
from facial images while preserving the person's appearance and facial details.

## Preprocessing Method
Deep Learning-based Background Removal using U²-Net.

## Input
Processed face images from User_01 and User_02.

## Processing Tasks
1. Load face images from the processed dataset.
2. Remove image backgrounds.
3. Generate images with transparent backgrounds.
4. Display 5 randomly selected images from each user.


## Output
- Background-removed images.
- Transparent PNG images.
- Visual comparison of preprocessing results.

## Purpose
Prepare cleaner images for the next stages:
Face Detection and Feature Extraction.


## 1. Import Libraries

In [ ]:
!pip install rembg
!pip install "rembg[cpu]"

In [ ]:
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import random

from pathlib import Path
from rembg import remove, new_session
from PIL import Image

In [ ]:
import os
import sys
from pathlib import Path

os.environ["U2NET_HOME"] = str(
    Path(sys.prefix) / "rembg_models"
)

print("Model directory:", os.environ["U2NET_HOME"])


## 2. Dataset Paths

Define the project directories and locate the processed face images
for each user.


In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent

PROCESSED_PATH = (
    PROJECT_ROOT / "Dataset" / "Processed" / "Face"
)

OUTPUT_PATH = (
    PROJECT_ROOT / "Dataset" / "Processed" / "Face_Background_Removed"
)

USERS = ["User_01", "User_02"]

print("Project Root:", PROJECT_ROOT)
print("Input Dataset:", PROCESSED_PATH)
print("Output Dataset:", OUTPUT_PATH)

print("\nDataset Exists:", PROCESSED_PATH.exists())

for user in USERS:
    user_path = PROCESSED_PATH / user

    print(
        f"{user}: {user_path.exists()}"
    )

## 2. Background Removal Setup

Initialize the U²-Net model session to prepare for background removal.

In [ ]:
from rembg import new_session

session = new_session("u2net")

print("U²-Net session initialized successfully.")

## 3. Background Removal Test



In [ ]:
import random
import matplotlib.pyplot as plt
from rembg import remove
from PIL import Image



fig, axes = plt.subplots(2, 5, figsize=(20, 9))

for user_index, user in enumerate(USERS):

    user_path = PROCESSED_PATH / user
    image_paths = list(user_path.glob("*"))

    selected_images = random.sample(image_paths, 5)

    for i, image_path in enumerate(selected_images):

        original = Image.open(image_path).convert("RGB")

        result = remove(
            original,
            session=session
        )

        axes[user_index, i].imshow(result)
        axes[user_index, i].set_title(
            f"{user}\n{image_path.name}"
        )
        axes[user_index, i].axis("off")

plt.tight_layout()
plt.show()

## 4. Full Dataset Background Removal

Apply U²-Net background removal to all accepted face images.

In [ ]:
from rembg import remove
from PIL import Image
from tqdm import tqdm

OUTPUT_PATH = PROJECT_ROOT / "Dataset" / "Processed" / "Face_Background_Removed"

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png"}

total_processed = 0
total_skipped = 0

for user in USERS:

    input_dir = PROCESSED_PATH / user
    output_dir = OUTPUT_PATH / user

    output_dir.mkdir(parents=True, exist_ok=True)

    image_paths = sorted([
        path for path in input_dir.iterdir()
        if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
    ])

    print(f"\nProcessing {user}: {len(image_paths)} images")

    for image_path in tqdm(image_paths, desc=user):

        output_file = output_dir / f"{image_path.stem}.png"

        # Skip images that have already been processed
        if output_file.exists():
            total_skipped += 1
            continue

        with Image.open(image_path) as img:
            input_image = img.convert("RGB")

        # Remove background using U²-Net
        output_image = remove(
            input_image,
            session=session
        )

        # Save transparent PNG
        output_image.save(output_file, format="PNG")

        total_processed += 1


print("\n" + "=" * 50)
print("BACKGROUND REMOVAL COMPLETED")
print("=" * 50)

print("Newly Processed Images:", total_processed)
print("Skipped Existing Images:", total_skipped)
print("Total Images:", total_processed + total_skipped)
print("Output Directory:", OUTPUT_PATH)

## 5. Image Standardization

### Step 1: Image Dimensions and Channels Inspection

Inspect the dimensions, color channels, and transparency
of background-removed images before resizing.

In [ ]:
from pathlib import Path
from PIL import Image
from collections import Counter

FACE_DIR = Path("../../Dataset/Processed/Face/Face_Background_Removed")

image_info = []

for user_folder in sorted(FACE_DIR.iterdir()):

    if not user_folder.is_dir():
        continue

    for image_path in sorted(user_folder.glob("*.png")):

        with Image.open(image_path) as img:

            image_info.append({
                "User": user_folder.name,
                "Filename": image_path.name,
                "Width": img.width,
                "Height": img.height,
                "Mode": img.mode,
                "Has_Alpha": "A" in img.getbands()
            })

print("Total Images:", len(image_info))

print("\nImage Modes:")
print(Counter(img["Mode"] for img in image_info))

print("\nImage Dimensions:")
print(Counter(
    (img["Width"], img["Height"])
    for img in image_info
))

print("\nTransparency Check:")
print(Counter(img["Has_Alpha"] for img in image_info))

print("\nLargest Images:")
for img in sorted(
    image_info,
    key=lambda x: x["Width"] * x["Height"],
    reverse=True
)[:5]:
    print(
        f'{img["Filename"]}: '
        f'{img["Width"]}x{img["Height"]} | '
        f'Mode: {img["Mode"]}'
    )

### Step 2: Image Resizing

In [ ]:
from pathlib import Path

# Input and output directories
INPUT_DIR = Path("../../Dataset/Processed/Face/Face_Background_Removed")

OUTPUT_DIR = Path("../../Dataset/Processed/Face/Face_Standardized_1024x1365")

# Target image dimensions
TARGET_WIDTH = 1024
TARGET_HEIGHT = 1365

# Create output directories
for user in ["User_01", "User_02"]:
    (OUTPUT_DIR / user).mkdir(parents=True, exist_ok=True)

print("Input Directory Exists:", INPUT_DIR.exists())
print("Output Directory:", OUTPUT_DIR.resolve())
print("Target Resolution:", f"{TARGET_WIDTH}x{TARGET_HEIGHT}")

In [ ]:
from PIL import Image

def calculate_resize_dimensions(image_width, image_height):

    scale = min(
        TARGET_WIDTH / image_width,
        TARGET_HEIGHT / image_height,
        1.0
    )

    new_width = round(image_width * scale)
    new_height = round(image_height * scale)

    return new_width, new_height

In [ ]:
test_dimensions = [
    (3472, 4624),
    (720, 960),
    (3056, 2614),
    (2762, 3177)
]

for width, height in test_dimensions:

    new_width, new_height = calculate_resize_dimensions(
        width, height
    )

    print(
        f"{width}x{height} -> "
        f"{new_width}x{new_height}"
    )

Step 2.2 — Resize Images

In [ ]:
from PIL import Image

processed_count = 0

for user_folder in sorted(INPUT_DIR.iterdir()):

    if not user_folder.is_dir():
        continue

    output_user_dir = OUTPUT_DIR / user_folder.name

    for image_path in sorted(user_folder.glob("*.png")):

        with Image.open(image_path) as img:

            original_width, original_height = img.size

            new_width, new_height = calculate_resize_dimensions(
                original_width,
                original_height
            )

            resized_img = img.resize(
                (new_width, new_height),
                Image.Resampling.LANCZOS
            )

            output_path = output_user_dir / image_path.name

            resized_img.save(output_path, format="PNG")

            processed_count += 1

print("Image Resizing Completed!")
print("Total Images Processed:", processed_count)

### Step 3: Transparent Padding

Place resized images inside a fixed-size transparent canvas
of 1024 × 1365 pixels.

Preserve the aspect ratio and transparency while centering
each image within the canvas.

In [ ]:
from PIL import Image

padded_count = 0

for user_folder in sorted(OUTPUT_DIR.iterdir()):

    if not user_folder.is_dir():
        continue

    for image_path in sorted(user_folder.glob("*.png")):

        with Image.open(image_path) as img:

            img = img.convert("RGBA")

            canvas = Image.new(
                "RGBA",
                (TARGET_WIDTH, TARGET_HEIGHT),
                (0, 0, 0, 0)
            )

            x_offset = (TARGET_WIDTH - img.width) // 2
            y_offset = (TARGET_HEIGHT - img.height) // 2

            canvas.alpha_composite(
                img,
                (x_offset, y_offset)
            )

            canvas.save(image_path, format="PNG")

            padded_count += 1

print("Transparent Padding Completed!")
print("Total Images Processed:", padded_count)

### Step 4: Standardized Images Validation

Verify that all standardized images have the target resolution,
RGBA color mode, and preserved transparency.

In [ ]:
from PIL import Image
from collections import Counter

validation_info = []

for user_folder in sorted(OUTPUT_DIR.iterdir()):

    if not user_folder.is_dir():
        continue

    for image_path in sorted(user_folder.glob("*.png")):

        with Image.open(image_path) as img:

            validation_info.append({
                "User": user_folder.name,
                "Filename": image_path.name,
                "Dimensions": img.size,
                "Mode": img.mode,
                "Has_Alpha": "A" in img.getbands()
            })

print("Total Images:", len(validation_info))

print("\nImage Dimensions:")
print(Counter(img["Dimensions"] for img in validation_info))

print("\nImage Modes:")
print(Counter(img["Mode"] for img in validation_info))

print("\nAlpha Channel:")
print(Counter(img["Has_Alpha"] for img in validation_info))

valid_images = sum(
    img["Dimensions"] == (TARGET_WIDTH, TARGET_HEIGHT)
    and img["Mode"] == "RGBA"
    and img["Has_Alpha"]
    for img in validation_info
)

print("\nValid Images:", valid_images)
print("Invalid Images:", len(validation_info) - valid_images)

### Step 4: Face Grayscale Standardization




In [ ]:
from pathlib import Path
import numpy as np
from PIL import Image

# Input and Output Paths
# The original project path was hard-coded to one computer. Run this notebook from the project root instead.
PROJECT_PATH = Path.cwd().resolve()

INPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Grayscale"
)

OUTPUT_PATH = (
    PROJECT_PATH
    / "Dataset"
    / "Processed"
    / "Face"
    / "Face_Grayscale_Standardization"
)

# Target Image Dimensions
TARGET_WIDTH = 256
TARGET_HEIGHT = 352

USERS = ["User_01", "User_02"]

# Create Output Directories
for user in USERS:
    (OUTPUT_PATH / user).mkdir(
        parents=True,
        exist_ok=True
    )

print("Output directories created successfully.")

In [ ]:
# Grayscale Image Standardization

standardization_results = {}

for user in USERS:

    standardization_results[user] = []

    user_input_path = INPUT_PATH / user
    user_output_path = OUTPUT_PATH / user

    images = sorted([
        path for path in user_input_path.iterdir()
        if path.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ])

    for image_path in images:

        # Load Grayscale image
        image = Image.open(image_path).convert("L")

        original_width, original_height = image.size

        # Calculate aspect-ratio-preserving scale
        scale = min(
            TARGET_WIDTH / original_width,
            TARGET_HEIGHT / original_height
        )

        new_width = round(original_width * scale)
        new_height = round(original_height * scale)

        # Resize image
        resized_image = image.resize(
            (new_width, new_height),
            Image.Resampling.LANCZOS
        )

        # Calculate padding color from image borders
        image_array = np.array(resized_image)

        border_pixels = np.concatenate([
            image_array[0, :],
            image_array[-1, :],
            image_array[:, 0],
            image_array[:, -1]
        ])

        padding_value = int(np.median(border_pixels))

        # Create standardized canvas
        standardized_image = Image.new(
            "L",
            (TARGET_WIDTH, TARGET_HEIGHT),
            color=padding_value
        )

        # Center resized face image
        paste_x = (TARGET_WIDTH - new_width) // 2
        paste_y = (TARGET_HEIGHT - new_height) // 2

        standardized_image.paste(
            resized_image,
            (paste_x, paste_y)
        )

        # Save standardized image
        output_path = user_output_path / image_path.name

        standardized_image.save(output_path)

        standardization_results[user].append(output_path)

    print(
        f"{user}: {len(standardization_results[user])} "
        "images standardized."
    )

print("\nGrayscale Standardization completed successfully.")